# Value-based RL Methods

## Value-based vs Policy-based Methods

### Policy-based

![policy-based](images/policy-based-diag.png)

### Value-based

![value-based](images/value-based-diag.png)

## Value Functions

### State-value Function
In an MDP, the state-value function $V^\pi:S -> \mathbb{R}$ is an estimate of the expected return starting from state $s$ and following policy $\pi$ thereafter.

$$V^\pi(s) = \mathbb{E}_{a_t \sim \pi(\cdot|s_t), s_{t+1} \sim P(\cdot|s_t, a_t)} [G_t | S_0 = s]$$

Denote the optimal state-value function as $V^*(s) = \max_\pi V^\pi(s)$

### Action-value Function

The action-value function $Q^\pi:S \times A -> \mathbb{R}$ is an estimate of the expected return starting from state $s$, taking action $a$, and following policy $\pi$ thereafter.

$$Q^\pi(s, a) = \mathbb{E}_{a_t \sim \pi(\cdot|s_t), s_{t+1} \sim P(\cdot|s_t, a_t)} [G_t | S_0 = s, A_0 = a]$$

Denote the optimal action-value function as $Q^*(s, a) = \max_\pi Q^\pi(s, a)$

### The relationship between $V^*$ and $\pi^*$

The state-value function and action-value function have a direct relationship

$$V^\pi(s) = \mathbb{E}_{a \sim \pi(\cdot|s)} [Q^\pi(s, a)]$$
$$Q^\pi(s, a) = \mathbb{E}_{s' \sim P(\cdot|s, a)} [R(s, a, s') + \gamma V^\pi(s')]$$

## The Bellman Equations

We can recursively express the value functions in terms of themselves using the Bellman equations:

$$V^\pi(s) = \mathbb{E}_{a \sim \pi, s'\sim P} [R(s, a, s') + \gamma V^\pi(s')]$$
$$V^*(s) = \max_a \mathbb{E}_{s'\sim P} [R(s, a, s') + \gamma V^*(s')]$$

$$Q^\pi(s,a) = \mathbb{E}_{s' \sim P(\cdot|s, a)} [R(s, a, s') + \gamma V^\pi(s')]$$
$$Q^*(s,a) = \mathbb{E}_{s' \sim P(\cdot|s, a)} [R(s, a, s') + \gamma \max_{a'} Q^*(s', a')]$$

## What can we do with the value functions?

- **Policy Evaluation**: Given a policy $\pi$, we can compute $V^\pi$ and $Q^\pi$ to evaluate the quality of the policy.
- **Poplicy Improvement**: Given $V^\pi$ or $Q^\pi$, we can improve the policy by acting greedily with respect to the value functions. For example, we can define a new policy $\pi'$ as follows:

This is the entire premise behind value-based control

![gpi](images/gpi.png)

- **Reducing Variance**: We can use value functions as baselines to reduce the variance of policy estimates (more on that in actor-critic methods).

## Exercise: Value function in FrozenLake

**Reminder from the previous tutorial**: [FrozenLake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) is a grid navigation environment with traps. The objective is to navigate from a start position to a goal position without falling through the ice in compromised cells.

- **Observation space**: cell position, calculated as `cur_row * n_cols + cur_col`
- **Action space**: up, down, left, right

For simplicity, let's make it deterministic by setting `is_slippery=False` when creating the environment.

In [ ]:
import gymnasium as gym
from PIL import Image

env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
env.reset()
frame = env.render()
Image.fromarray(frame)

### Reward Map

![rewards-map](images/grid-rewards.png)

### Go-right Policy $\pi_{\text{right}}$

![go-right](images/go-right-policy.png)

_Question_: What is the value of each state

### Go-right Value Function $V^{\pi_{\text{right}}}$

![right-vf](images/right-vf.png)

_Question_: What would the next policy look like, assuming we improve the policy greedily with respect to $V^{\pi_{\text{right}}}$? Assume tie-breaking in favor of the existing action.

### Improved Policy

![improved-policy](images/improved-policy.png)

### Optimal Value Function $V^*$

![optimal-vf](images/opt-vf.png)

## Value Iteration

![value-iteration](https://lcalem.github.io/imgs/sutton/value_iteration.png)

### VI in python for gymnasium toy-text environments

FrozenLake belongs to a class of problems known as "toy-text". One unifying feature of these environments is that they have a `P` attribute that contains the transition dynamics of the environment.

In [ ]:
# all transitions from state 1
env.unwrapped.P[1]

We can use this attribute to implement value iteration verbatim from the pseudocode:

In [ ]:
def value_iteration(env, gamma=0.9, theta=1e-5):
    # initialize VF arbitrarily to all 0
    v = [0] * env.observation_space.n

    # set delta to infinity for the first while condition check
    delta = float("inf")

    # iterate until convergence
    while delta > theta:
        delta = 0

        # iterate all states
        for s in range(env.observation_space.n):
            vs_old = v[s]  # save old state for convergence check

            # update VF for state s by taking the max over all actions of the expected return
            v[s] = max(
                # expectation over next states for action a
                sum(
                    p * (r + gamma * v[s_])
                    for p, s_, r, _ in env.unwrapped.P[s][a]
                )

                # iterate all actions for state s
                for a in range(env.action_space.n)
            )

            # update delta for convergence check
            delta = max(delta, abs(vs_old - v[s]))

    # extract greedy policy from optimal VF
    pi = {
        # select an action for each state
        s: max(
            # find maximal action
            range(env.action_space.n),

            # sort by expected return
            key=lambda a: sum(
                p * (r + gamma * v[s_]) for p, s_, r, _ in env.unwrapped.P[s][a]
            ),
        )
        for s in range(env.observation_space.n)
    }

    return pi, v


## VI in FrozenLake

In [ ]:
import numpy as np

pi, v = value_iteration(env)

np.array(v).reshape((4, 4))

The value function looks identical to what we computed earlier. Let's see if it solves the environment optimally as expected

In [ ]:
import mediapy as media

term = trun = False
obs, info = env.reset()
frames = [env.render()]
while not (term or trun):
    action = pi[obs]
    obs, reward, term, trun, info = env.step(action)
    frames.append(env.render())

media.show_video(frames, fps=3)

### VI in Slippery FrozenLake

We already knew how to solve the deterministic FrozenLake using classical planners, which are typically more efficient. But VI also supports stochastic action outcomes.

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=True, success_rate=0.8, render_mode='rgb_array')
pi, v = value_iteration(env)
np.array(v).reshape((4, 4))

It can now take longer to complete the task, and we can no longer guarantee that it will succeed every time. This is reflected in the value function. Let's run the policy several times to see how it does.

In [ ]:
from tqdm.auto import tqdm

N = 100_000

def run_fl(env, pi, n_episodes, pbar=True):
    wins = 0
    for _ in tqdm(range(n_episodes), disable=not pbar):
        term = trun = False
        obs, info = env.reset()
        while not (term or trun):
            action = pi[obs]
            obs, reward, term, trun, info = env.step(action)
        wins += reward
    return wins

wins = run_fl(env, pi, n_episodes=N)
print(f"Wins: {wins}/{N} ({wins / N:.1%})")

## Model-free Methods

_Question_: Is VI a planning algorithm or an RL algorithm?

It is mostly a planning algorithm, since it uses a model of the environment to compute the optimal policy. It does not follow the observe-act-learn loop of typical RL algorithms.

![observe-act-learn](images/act-observe.png)

However, it also computes a value function and a policy that can be executed in the environment, which is what RL algorithms do. So it has elements of both.

Still, VI relies on an accurate model of the environment, which is not always available. A **model-free** method is one that does not require a model of the environment to learn a policy. Instead, it learns directly from interactions with the environment, following the observe-act-learn loop exactly.

## Monte Carlo Learning

Monte Carlo methods, as the name suggests, rely on sampling to estimate value functions. They learn from complete episodes of experience, using the returns observed at the end of each episode to update value estimates.

![mc](https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/MC-5.jpg)

This is completely model-free: instead of using a model to compute expected returns, it uses actual returns observed from interactions with the environment.

Pros:
- **Unbiased**: Relies on actual returns, so it is an unbiased estimator of the true value function.
- **Targeted**: Easy to learn from specific states without needing to evaluate the rest of the state space.
- **Does not rely on the Markov Property**: Can be applied to non-Markovian environments (can handle partial observability).

Cons:
- **High Variance**: Returns can be noisy, leading to high variance in updates.
- **Episodic**: Requires complete episodes, which can be inefficient for long episodes or continuing tasks.
- **Slow Convergence**: Can take a long time to converge, especially in environments with long episodes.

## Monte Carlo Control

Sample full episode trajectories with an $\epsilon$-greedy policy to estimate the action-value function $Q^\pi$, and then improve the policy by acting greedily with respect to $Q^\pi$.

![monte-carlo-conrol](https://lcalem.github.io/imgs/sutton/onpolicy_first_visit_mc.png)

_Question_: Is this an on-policy or off-policy method?

_Question_: Where is the learning rate $\alpha$ in this algorithm?

### Convergence

Under strict conditions, the policy is guaranteed to converge to $Q^*$, and thus to the optimal policy $\pi^*$:
- **Finite MDP**: Finite state and action spaces.
- **Infinite Exploration**: Every state-action pair must be visited infinitely often
- **Bounded Rewards**: The rewards must be bounded to ensure that the returns are well-defined.
- **Appropriate Step-size**: If using a constant step-size $\alpha$, it must satisfy the Robbins-Monro conditions (see [Watkins & Dayan, 1992](https://link.springer.com/article/10.1007/BF00992698)).
- **Stationary Environment**: The environment's dynamics must not change over time.
- **Greedy in the Limit with Infinite Exploration (GLIE)**: The policy must eventually become greedy with respect to the action-value estimates, e.g., through $\epsilon$-decay.

_Question_: Do these conditions hold for the implementation described in the pseudocode?

## MC Control in FrozenLake

In [ ]:
def evaluate_policy(env, pi, n_episodes):
    wins = run_fl(env, pi, n_episodes=n_episodes, pbar=False)
    return wins / n_episodes

def episilon_soft_policy(Q, epsilon, n_actions):
    # greedy action for each state with random tie-breaking
    greedy_pi = np.array([
        np.random.choice(np.flatnonzero(Q[s] == Q[s].max()))
        for s in range(Q.shape[0])
    ])

    return {
        s: [
            epsilon / n_actions if a != greedy_pi[s] else 1 - epsilon + epsilon / n_actions
            for a in range(n_actions)
        ]
        for s in range(env.observation_space.n)
    }

def monte_carlo_control(env, n_episodes, gamma=0.9, epsilon=0.1, n_eval_episodes=100, sticky_pbar=True):
    # Initialize empty action-value function, empty returns, and random epsilon-soft policy,
    # NOTE: The algorithm calls for Q-value 0 at terminal states, but this in practice, a random
    #       initialization often works better than 0, which can lead to no exploration.
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    returns = {s: {a: [] for a in range(env.action_space.n)} for s in range(env.observation_space.n)}
    pi = episilon_soft_policy(Q, epsilon, env.action_space.n)

    # accumulate rewards for each episode for monitoring
    eval_scores = []
    q_history = []  # full Q after each episode, for bias-variance plots
    for _ in tqdm(range(n_episodes), leave=sticky_pbar):
        # generate an episode following current epsilon-greedy policy
        episode = []
        obs, info = env.reset()
        terminated = truncated = False
        while not (terminated or truncated):
            a = np.random.choice(env.action_space.n, p=pi[obs])
            next_obs, reward, terminated, truncated, info = env.step(a)
            episode.append((obs, a, reward))
            obs = next_obs

        # first-visit MC: update returns and Q
        G = 0.0
        visited_sa = set()
        for s, a, r in reversed(episode):
            G = gamma * G + r
            if (s, a) not in visited_sa:
                visited_sa.add((s, a))
                returns[s][a].append(G)
                Q[s, a] = np.mean(returns[s][a])
        
        eval_scores.append(evaluate_policy(env, Q.argmax(axis=1), n_eval_episodes))
        q_history.append(Q.copy())

        # update policy to be epsilon-soft w.r.t. new Q
        pi = episilon_soft_policy(Q, epsilon, env.action_space.n)

    return pi, Q, eval_scores, q_history

### Training

In [ ]:
soft_pi, Q, mcc_eval_scores, _ = monte_carlo_control(env, n_episodes=1000)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(mcc_eval_scores)
plt.xlabel("Iterations")
plt.ylabel("Success Rate")

### Evaluation

In [ ]:
pi = Q.argmax(axis=1)
wins = run_fl(env, pi, n_episodes=N)
print(f"Wins: {wins}/{N} ({wins / N:.1%})")

## Temporal Difference Learning

In temporal difference learning, or TD-learning, we learn the value function by bootstrapping from the current estimates of the value function, rather than waiting for complete episodes as in Monte Carlo methods.

![td](https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/TD-3.jpg)

Pros:
- **Lower Variance**: By bootstrapping, TD methods can have lower variance than Monte Carlo methods, since they do not rely on complete returns.
- **Online Learning**: TD methods learn at every step, without needing to wait for the end of an episode.
- **Handles continuing tasks**: The online nature of TD methods makes them suitable for long or never-ending tasks.

Cons:
- **High Bias**: Bootstrapping can introduce bias, since it relies on estimates of the value function rather than actual returns.
- **Slow Propagation for Sparse Rewards**: If rewards are sparse, it can take a long time for value estimates to propagate back to earlier states.
- **Relies on the Markov Property**: The TD learning update rule leans heavily on the environment being Markovian.

## Q-learning

A common TD control algorithm is Q-learning. Like Monte Carlo control, it learns the optimal action-value function $Q^*$ directly by updating estimates based on the observed reward and the maximum estimated value of the next state. Unlike Monte Carlo control, it does so at each step using the TD target and the current Q-value estimate.

![Q-learning](https://lcalem.github.io/imgs/sutton/qlearning.png)

_Question_: Is this an on-policy or off-policy method?

Convergence is guaranteed under the same conditions to Monte Carlo control, but since it is off-policy, it can converge even if the behavior policy is not GLIE.

_Question_: Do these conditions hold for the implementation described in the pseudocode?

## Q-learning in FrozenLake

In [ ]:
def q_learning(env, n_episodes, gamma=0.9, alpha=0.01, epsilon=0.1, n_eval_episodes=100, sticky_pbar=True):
    # Initialize empty action-value function
    # NOTE: The algorithm calls for Q-value 0 at terminal states, but this in practice, a random
    #       initialization often works better than 0, which can lead to no exploration.
    Q = np.zeros((env.observation_space.n, env.action_space.n))

    n_sa = {s: {a: 0 for a in range(env.action_space.n)} for s in range(env.observation_space.n)}

    # accumulate rewards for each episode for monitoring
    eval_scores = []
    q_history = []  # full Q after each episode, for bias-variance plots
    for _ in tqdm(range(n_episodes), leave=sticky_pbar):
        obs, info = env.reset()
        terminated = truncated = False
        while not (terminated or truncated):
            if np.random.rand() < epsilon:
                a = env.action_space.sample()  # explore
            else:
                a = np.random.choice(np.flatnonzero(Q[obs] == Q[obs].max()))  # exploit with random tie-breaking
            next_obs, reward, terminated, truncated, info = env.step(a)

            n_sa[obs][a] += 1

            # Q-learning update
            Q[obs, a] += alpha * (reward + gamma * Q[next_obs].max() - Q[obs, a])

            obs = next_obs

        eval_scores.append(evaluate_policy(env, Q.argmax(axis=1), n_eval_episodes))
        q_history.append(Q.copy())

    return Q, eval_scores, q_history

### Training

In [ ]:
Q, ql_eval_scores, _ = q_learning(env, n_episodes=1000)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(ql_eval_scores, label="Q-Learning")
plt.xlabel("Iterations")
plt.ylabel("Success Rate")
plt.legend()
plt.show()

### Evaluation

In [ ]:
pi = Q.argmax(axis=1)
wins = run_fl(env, pi, n_episodes=N)
print(f"Wins: {wins}/{N} ({wins / N:.1%})")

## Comparing Monte Carlo Control and Q-learning

Let's execute multiple runs of both algorithms and compare their performance.

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    env.reset(seed=seed)

N_EPISODES = 2000

seeds = [42 * i for i in range(10)]
mc_eval_scores_runs = []
ql_eval_scores_runs = []
mc_q_history_runs = []
ql_q_history_runs = []
for seed in tqdm(seeds):
    set_seed(seed)
    _, _, mcc_eval_scores, mcc_q_history = monte_carlo_control(env, n_episodes=N_EPISODES, sticky_pbar=False)
    mc_eval_scores_runs.append(mcc_eval_scores)
    mc_q_history_runs.append(mcc_q_history)

    set_seed(seed)
    _, ql_eval_scores, ql_q_history = q_learning(env, n_episodes=N_EPISODES, sticky_pbar=False)
    ql_eval_scores_runs.append(ql_eval_scores)
    ql_q_history_runs.append(ql_q_history)
    

### Success rate over time

_Question_: Which method's policy do we expect to converge faster?

In [ ]:
# plot mean and std
plt.plot(np.mean(mc_eval_scores_runs, axis=0), label="Monte Carlo Control")
plt.fill_between(range(N_EPISODES),
                 np.mean(mc_eval_scores_runs, axis=0) - np.std(mc_eval_scores_runs, axis=0),
                 np.mean(mc_eval_scores_runs, axis=0) + np.std(mc_eval_scores_runs, axis=0),
                 alpha=0.2)
plt.plot(np.mean(ql_eval_scores_runs, axis=0), label="Q-Learning")
plt.fill_between(range(N_EPISODES),
                 np.mean(ql_eval_scores_runs, axis=0) - np.std(ql_eval_scores_runs, axis=0),
                 np.mean(ql_eval_scores_runs, axis=0) + np.std(ql_eval_scores_runs, axis=0),
                 alpha=0.2)
plt.xlabel("Iterations")
plt.ylabel("Success Rate")
plt.legend()
plt.show()

_Question_: What are these jumps in the Q-learning curve?

## Convergence to Q*

If we look at the value of $Q^\pi(s, a^*)$ for the optimal action $a^* = \arg\max_a Q^*(s, a)$, we can see how each method's estimates converge to the optimal value $Q^*(s, a^*) = V^*(s)$ across seeds. This shows the bias-variance tradeoff between the two methods.

In [ ]:
from ipywidgets import interact

ACTIONS = {
    0: "Left",
    1: "Down",
    2: "Right",
    3: "Up",
}

# reference: a*(s) = argmax_a Q*(s, a) from VI, Q*(s, a*) = V*(s)
vi_pi, vi_v = value_iteration(env)

mc_q_history_arr = np.asarray(mc_q_history_runs)  # (n_seeds, n_episodes, n_states, n_actions)
ql_q_history_arr = np.asarray(ql_q_history_runs)

@interact(s=(0, env.observation_space.n - 1))
def plot_q_estimate(s=0):
    a_star = vi_pi[s]
    q_star = vi_v[s]

    fig, axes = plt.subplots(1, 2, sharey=True, figsize=(11, 4))
    fig.suptitle(f"Q(s={s}, a*={ACTIONS[a_star]}) estimate across seeds")

    for ax, runs, title, color in [
        (axes[0], mc_q_history_arr, "Monte Carlo Control", "C0"),
        (axes[1], ql_q_history_arr, "Q-Learning", "C1"),
    ]:
        series = runs[:, :, s, a_star]  # (n_seeds, n_episodes)
        for r in series:
            ax.plot(r, alpha=0.25, color=color)
        ax.plot(series.mean(axis=0), color=color, linewidth=2, label="mean across seeds")
        ax.axhline(q_star, linestyle="--", color="k", label=fr"$Q^*(s_{{{s}}}, a^*)$")
        ax.set_title(title)
        ax.set_xlabel("Episode")
        ax.legend(loc="lower right")

    axes[0].set_ylabel(fr"$Q(s_{{{s}}}, a^*)$ estimate")
    plt.tight_layout()
    plt.show()

Numbered states in the grid for reference:

![numbered-states-on-grid](images/numbered-states.png)

## Conclusion

In this tutorial, we explored value-based methods for solving RL problems.
- We defined the state-value $V^\pi$ and action-value $Q^\pi$ functions.
- We saw how the Bellman equations enable recursive value computation.
- We solved FrozenLake using value iteration as a model-based planner.
- We learned $Q^*$ in a model-free way using Monte Carlo control.
- We applied Q-learning, an off-policy TD control method.
- We compared MC and Q-learning empirically, observing the bias-variance tradeoff.

Value-based RL is a powerful and well-understood family of methods.
- It is model-free and learns directly from interaction with the environment.
- It comes with strong convergence guarantees under standard conditions.
- TD methods are sample efficient through bootstrapping.
- Off-policy methods like Q-learning can learn from arbitrary behavior data.
- The learned $Q$ function gives you a policy for free via $\arg\max_a Q(s, a)$.

But value-based RL also has many limitations.
- Tabular methods do not scale. One entry per $(s, a)$ pair quickly becomes infeasible.
- Hyperparameters (learning rate, exploration, decay schedule) require careful tuning.
- The bias-variance tradeoff between MC and TD is real and unavoidable.
- Sparse rewards make value propagation slow, especially for distant states.
- $\arg\max$-based control is awkward for continuous action spaces.

In the next tutorial, we will use neural networks as Q-function approximators to overcome the scalability issues of tabular methods and apply value-based RL to more complex environments.